In [ ]:
import os
import json
import time
import random
import requests
from pathlib import Path
HOME = "https://x-fetish.tube/"

json_dir = Path(r"C:\Users\rando\Office Projects\rep_fsparse\json")

for json_path in json_dir.glob("*.json"):
    

    
    path = Path(json_path)
    
    if not path.stem.startswith("e"):
        continue

    OUT = fr"D:\movies\{path.stem}"
    os.makedirs(OUT, exist_ok=True)

    with open(json_path, "r", encoding="utf-8") as f:
        links = json.load(f)

    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/153.0.0.0 Safari/537.36"
        ),
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    })

    r = session.get(HOME, timeout=30)
    r.raise_for_status()

    print("Session created")
    print("Cookies:", session.cookies.get_dict())
    print(f"Downloading {len(links)} images...\n")

    for i, url in enumerate(links, 1): 
        

        
        path = os.path.join(OUT, f"{i:05d}.jpg")

        if os.path.exists(path):
            print(f"[{i}/{len(links)}] EXISTS")
            continue

        try:
            r = session.get(
                url,
                headers={"Referer": HOME},
                timeout=30
            )

            if r.status_code == 200:
                with open(path, "wb") as f:
                    f.write(r.content)

                print(f"[{i}/{len(links)}] OK")

            else:
                print(f"[{i}/{len(links)}] HTTP {r.status_code}")

        except requests.RequestException as e:
            print(f"[{i}/{len(links)}] FAILED: {e}")

        time.sleep(random.uniform(0.3, 0.5))

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import json5,json
import fitz #type: ignore
import json
import csv
from typing import Union, Dict, Any, List

from app.logger import *
from app.amc.fund_data import *
from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

utils = Helper()

def json_to_csv(input_data: str,output_file: str,spacing: int = 2) -> int:
    """
    Convert mutual fund JSON into a structured CSV.
    Parameters:
        output_file (str): Path to output CSV file
        spacing (int): Number of empty rows after each mutual fund
    Returns:
        int: Total number of rows written (excluding header)
    Raises:
        ValueError: If input data format is invalid
        IOError: If file operations fail
    """
    try:
        with open(input_data, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        raise

    # Validate
    if "records" not in data or not isinstance(data["records"], list):
        raise ValueError("Invalid JSON structure")

    row_count = 0
    try:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "mutual_fund_name",
                "main_scheme_name",
                "sfin",
                "monthly_aum_value",
                "aum_in_mil", #new
                "portfolio_data_0",
                "portfolio_data_1",
                "page",
                "table",
            ])

            # Process each record
            for record in data["records"]:
                value = record.get("value", {})
                mf_name = value.get("mutual_fund_name", "")
                scheme_name = value.get("main_scheme_name", "")
                aum_value = value.get("monthly_aaum_value", "")
                portfolio_list: List[Dict[str, Any]] = value.get("portfolio_data", [])
                sfin = value.get("sfin","")
                
                #create value
                try:
                    aum_in_mil = round(float(aum_value)*10,2)
                
                except:
                    aum_in_mil = ""

                # Skip if no portfolio data
                if not isinstance(portfolio_list, list):
                    continue

                for item in portfolio_list:
                    writer.writerow([
                        mf_name,
                        scheme_name,
                        sfin,
                        aum_value,
                        aum_in_mil,
                        item.get("0", ""),
                        item.get("1", ""),
                        item.get("page", ""),
                        item.get("table", ""),
                    ])
                    row_count += 1

                # Add spacing rows
                for _ in range(spacing):
                    writer.writerow([])
        return row_count
    except Exception as e:
        raise IOError(f"Error writing CSV: {e}")

In [ ]:
#INSURANCE FUND

amc_id = "85_0"
path = r"85_31-May-26_IF.pdf"

config = get_config("2026",amc_id)
regex = get_regex("2026")

object = HDFCLifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)

In [4]:
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [5]:
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
    
save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
    
print(f"File Saved At: {save_path}")



File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\82_31-May-26_IF.json


In [6]:
output_path = save_path.replace(".json",".csv")
json_to_csv(save_path,output_path)

3948

In [ ]:
pattern = "Rs.\\s*([\\d,.]+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith(".aum"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)